In [ ]:
# %% [markdown]
# # Retrieval-Augmented Generation (RAG) Pipeline
# This notebook implements a RAG system using LangChain, ChromaDB, and FLAN-T5.

# %%
# 1. Installation
# Install required libraries
%pip install -q -U bitsandbytes transformers accelerate langchain chromadb langchain-huggingface datasets python-dotenv langchain-text-splitters langchain-community

# IMPORTANT: After this cell finishes, go to Runtime -> Restart Session
# %%
import os
import time
from datasets import load_dataset
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch


In [ ]:


# %% [markdown]
# ## 2. Data Loading & Preprocessing
# We use the IBM WatsonX documentation dataset.

# %%
# Load the IBM dataset
corpus_dataset = load_dataset("ibm-research/watsonxDocsQA", "corpus", split="train")
print(type(corpus_dataset[0]))

# Convert into LangChain Document objects
documents = []
source_dict ={}
for entry in corpus_dataset:
    content = f"Title: {entry['title']}\n\n{entry['document']}"
    doc = Document(page_content=content, metadata={"source": entry['doc_id']})
    source_dict[entry['doc_id']] = content
    documents.append(doc)

print(f"Loaded {len(documents)} documents.")

# Split documents into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=300)
docs = text_splitter.split_documents(documents)
print(f"Split into {len(docs)} chunks.")









In [ ]:
# %% [markdown]
# ## 3. Vector Database Construction
# Using sentence-transformers to create embeddings and ChromaDB for storage.

# %%
# Initialize embeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

# Process in batches to manage memory
batch_size = 100 
vectorstore = None

print(f"Starting embedding for {len(docs)} chunks...")

for i in range(0, len(docs), batch_size):
    batch = docs[i : i + batch_size]
    if vectorstore is None:
        vectorstore = Chroma.from_documents(documents=batch, embedding=embeddings)
    else:
        vectorstore.add_documents(batch)
    print(f"✅ Completed {i + len(batch)} / {len(docs)} chunks...")

# Define the retriever with a threshold
# retriever = vectorstore.as_retriever(
#     search_type="mmr",
#     search_kwargs={"score_threshold": 0.2, "k": 3}
# )



In [ ]:

# retriever = vectorstore.as_retriever(
#     search_type="mmr",
#     search_kwargs={"score_threshold": 0.2, "k": 5}
# )
# Updated Retriever with MMR + Similarity integration
# retriever = vectorstore.as_retriever(
#     search_type="mmr",
#     search_kwargs={
#         "k": 10,                # Final number of chunks passed to the Re-ranker
#         "fetch_k": 50,          # Initial pool fetched via Similarity Search
#         "lambda_mult": 0.8    # 1.0 is pure similarity, 0.0 is max diversity
#     }
# )


In [ ]:
import os
from dotenv import load_dotenv
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import BitsAndBytesConfig

# Load variables from .env
load_dotenv() 
hf_token = os.getenv("HF_TOKEN")

model_id = "mistralai/Mistral-7B-Instruct-v0.2"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype="float16",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)
# Now the library uses the token from your .env file
tokenizer = AutoTokenizer.from_pretrained(model_id, token=hf_token)
model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    quantization_config=bnb_config, 
    device_map="auto", 
    token=hf_token
)



In [ ]:

retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 15,                # Final number of chunks passed to the Re-ranker
        "fetch_k": 50,          # Initial pool fetched via Similarity Search
        "lambda_mult": 0.6     # 1.0 is pure similarity, 0.0 is max diversity
    }
)
# retriever = vectorstore.as_retriever(
#     search_type="similarity",
#     search_kwargs={
#         "k": 5,                # Final number of chunks passed to the Re-ranker
#            # 1.0 is pure similarity, 0.0 is max diversity
#     }
# )

In [ ]:
from sentence_transformers import CrossEncoder

# Initialize the Re-ranker (BGE-Reranker is highly effective for technical docs)
rerank_model = CrossEncoder('BAAI/bge-reranker-base', device="cpu")

def rag_pipeline(question):
    # 1. Retrieve: Initial candidates (Similarity + MMR)
    initial_docs = retriever.invoke(question)
    if not initial_docs:
        return "No relevant documents found."

    # 2. Re-rank: Use Cross-Encoder to score the MMR results
    # Create pairs of [Question, Chunk Text]
    pairs = [[question, doc.page_content] for doc in initial_docs]
    scores = rerank_model.predict(pairs)
    
    # Sort docs by the new re-ranker scores
    scored_docs = sorted(zip(scores, initial_docs), key=lambda x: x[0], reverse=True)
    
    # Select top 3 after re-ranking to reduce LLM "distraction"
    top_docs = [doc for score, doc in scored_docs[:3]]

    # 3. Format context
    context = "\n\n".join([doc.page_content for doc in top_docs])

    # 4. Generate with Mistral
    messages = [
        {"role": "user", "content": f"Context: {context}\n\nQuestion: {question}\n\nAnswer using only the provided context."}
    ]
    
    inputs = tokenizer.apply_chat_template(
        messages, 
        add_generation_prompt=True, 
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs, 
            max_new_tokens=400, 
            temperature=0.1, # Keep low to prevent hallucinations
            pad_token_id=tokenizer.eos_token_id
        )

    prompt_length = inputs['input_ids'].shape[-1] 
    return tokenizer.decode(outputs[0][prompt_length:], skip_special_tokens=True)

In [ ]:
question = "What foundation models have been built by IBM?"
print(f"Question: {question}")
print("-" * 30)
print(f"Answer: {rag_pipeline(question)}")

In [ ]:
# %%
# Debug function to see what the retriever is actually seeing
def debug_retrieval(question):
    results = retriever.invoke(question)
    print(f"Found {len(results)} relevant chunks.")
    for i, res in enumerate(results):
        
        print(f"\n--- Chunk {i+1} ---\n{res.page_content}...")
        print(res.metadata)

debug_retrieval(question)



In [ ]:
from datasets import load_dataset

datset_test = load_dataset(
    "ibm-research/watsonxDocsQA",
    "question_answers",
    split="train"
)
print(len(datset_test),"length")


# -----------------------------
# FIXED DEBUG FUNCTION
# -----------------------------
def debug_retrieval(question, true_doc_id, k=3):
    results = retriever.invoke(question)

    print("\n====================================")
    print(f"❓ Question: {question}")
    print(f"🎯 True Doc ID: {true_doc_id}")
    print("====================================")

    match_found = False

    for i, res in enumerate(results[:k]):
      
        pred_id = res.metadata.get("source")

        match = (pred_id == true_doc_id)

        if match:
            match_found = True

        print(f"\n--- Chunk {i+1} ---")
        print("📌 Predicted Doc ID:", pred_id)
        print("✅ MATCH:", match)
        print(res.page_content[:500])

    return match_found


# -----------------------------
# RUN ON 20 ROWS
# -----------------------------
results = []

for row in datset_test.select(range(45)):
    question = row["question"]
    true_doc_id = row["correct_answer_document_ids"]

    match = debug_retrieval(question, true_doc_id, k=3)

    results.append(match)


# -----------------------------
# FINAL SCORE
# -----------------------------
hit_at_3 = sum(results) / len(results)

print("\n====================")
print("📊 FINAL RESULT")
print("====================")
print("Hit@3 Accuracy:", hit_at_3)

In [ ]:
import torch
from sentence_transformers import CrossEncoder

# 1. Setup the Reranker (Ensuring it matches your pipeline)
rerank_model = CrossEncoder('BAAI/bge-reranker-base', device="cpu")

# 2. Updated Debug Function that uses the Reranker
def debug_retrieval_with_reranker(question, true_doc_id, k=3):
    # Get 15 initial candidates (to give the reranker a good pool)
    initial_docs = retriever.invoke(question)
    
    if not initial_docs:
        print(f"⚠️ No docs found for: {question[:30]}...")
        return False, False

    # Create pairs for the Cross-Encoder [Question, Chunk Content]
    pairs = [[question, doc.page_content] for doc in initial_docs]
    
    # Calculate scores and sort
    scores = rerank_model.predict(pairs)
    scored_docs = sorted(zip(scores, initial_docs), key=lambda x: x[0], reverse=True)
    
    # Take the top k results after reranking
    top_reranked_docs = [doc for score, doc in scored_docs[:k]]

    hit_1 = False
    hit_3 = False
    
    print("\n" + "="*50)
    print(f"❓ Question: {question}")
    print(f"🎯 target ID: {true_doc_id}")
    print("-" * 20)

    for i, res in enumerate(top_reranked_docs):
        pred_id = str(res.metadata.get("source")).strip()
        clean_true_id = str(true_doc_id).strip()
        
        is_match = (pred_id == clean_true_id)
        
        if is_match:
            if i == 0: hit_1 = True
            hit_3 = True
        
        # Print status for each of the top 3
        print(f"Rank {i+1} | ID: {pred_id} | Match: {is_match}")

    return hit_1, hit_3

# 3. Execution Loop (Test 40 rows)
results_h1 = []
results_h3 = []

print("🚀 Starting Evaluation...")

# Assuming datset_test is your loaded 'ibm-research/watsonxDocsQA'
for row in datset_test.select(range(40)):
    q = row["question"]
    target = row["correct_answer_document_ids"]
    
    h1, h3 = debug_retrieval_with_reranker(q, target, k=3)
    
    results_h1.append(h1)
    results_h3.append(h3)

# 4. Final Statistics
final_h1 = sum(results_h1) / len(results_h1)
final_h3 = sum(results_h3) / len(results_h3)

print("\n" + "📊" * 15)
print("      FINAL ACCURACY REPORT")
print("📊" * 15)
print(f"Hit@1 (Top Result): {final_h1:.2%}")
print(f"Hit@3 (In Top 3):   {final_h3:.2%}")
print("=" * 30)

if final_h1 > 0.85:
    print("\n✅ DECISION: Implement 'Pass Full Document'.")
    print("Since your Hit@1 is high, the top ID is reliable. ")
    print("Feeding that full document to the LLM will solve the 'Missing List' issue.")